# Proyecto PBL

Este Notebook realiza un preprocesamiento de los datos y automatiza los procesos:
- "Procesamiento eficiente",

de acuerdo con:
https://unilever-my.sharepoint.com/:x:/r/personal/antonio_arteaga_unilever_com/Documents/PBL/Logica%20Procesamiento%20eficiente%20by%20Rodrigo.xlsx?d=w448e0c42945c4baeb42502cd083e06c7&csf=1&web=1&e=WlB2n3


## Preprocesamiento de los datos

In [0]:
from datetime import datetime, date, timedelta
from dateutil.relativedelta import relativedelta
import pyspark.sql.functions as F
import pandas as pd
from pyspark.sql.types import StringType,DoubleType,LongType,BooleanType,ArrayType,IntegerType
import re, glob, pytz
from functools import reduce
from pyspark.sql import DataFrame, Window

Se usara el catalogo 'hive_metastore'. 'spark_catalog' es equivalente a 'hive_metastore'.

In [0]:
%sql
SELECT current_catalog()

current_catalog()
spark_catalog


### Extraccion de datos

In [0]:

flujo_one = spark.sql("""
SELECT *
FROM dwh_mx_unilever_status_one.flujo_one
WHERE Canal = "01"
    AND Clase_DocVentas = "ZOR"
    AND CategItem = "ZTAN"
    -- AND Cantidad_Orden > 0
    --AND Org_Ventas = 'MX01'
    --AND Desc_Solicitante IN ('DESARROLLO COML ABARROTERO SA DE CV', 'DECASA DEL CENTRO S A DE C V')
    --AND Desc_Solicitante = "CASA LEY S A P I DE C V"
    --AND Solicitante = "10134611"
    --AND Material = "62694482"
    AND `Fecha_de_Creación_del_Pedido` >= DATE '2025-01-01'
    AND `Fecha_de_Creación_del_Pedido` <= DATE '2027-01-31'
    --AND `Fecha_de_Creación_del_Pedido` = DATE '2026-05-18'
""")

# Maestro de matriales - se descarga de SAP automaticamente
marm = spark.sql("SELECT * FROM mx_unilever_pbl.marm")
marm = marm.dropDuplicates(subset = ["Material"])

In [0]:
solicitantes = spark.sql("SELECT * FROM mx_unilever_pbl.solicitantes")
grupos = spark.sql("SELECT * FROM mx_unilever_pbl.grupos")
menus = spark.sql("SELECT * FROM mx_unilever_pbl.menus")
perfil_de_flota = spark.sql("SELECT * FROM mx_unilever_pbl.perfil_de_flota")
others = spark.sql("SELECT * FROM mx_unilever_pbl.others")
excepcion = spark.sql("SELECT * FROM mx_unilever_pbl.excepcion")
motivos_rechazo = spark.sql("SELECT * FROM mx_unilever_pbl.motivos_rechazo")

In [0]:
motivos_rechazo.printSchema()

root
 |-- ID: long (nullable = true)
 |-- MOTIVO: string (nullable = true)
 |-- CONCEPTO: string (nullable = true)



In [0]:
# DataFrame shape
print((flujo_one.count(), len(flujo_one.columns)))

(3726243, 61)


In [0]:
# Show DataFrame schema
# flujo_one.printSchema()

### Preprocesamiento

In [0]:
# Agregar periodo
flujo_one = (flujo_one
.withColumn(
    "YYYY_MM",
    F.date_format(F.col("Fecha_de_Creación_del_Pedido"), "yyyy.MM")
))

# Cambiar nombre de columnas
flujo_one = (
    flujo_one
    .withColumnRenamed("Cantidad_Alocada", "Del_Qty")
    .withColumnRenamed("Cantidad_Facturada", "Inv_Qty")
    .withColumnRenamed("Ordenadas", "Ord_Qty")
    .withColumnRenamed("Cantidad_Orden", "Orig_Qty")
    .withColumnRenamed("GSV_X_CAJA", "CS_GSV")
)

# Quitar motivos a excluir
flujo_one = (
    flujo_one.alias("A")
    .join(
        motivos_rechazo.select("MOTIVO").dropDuplicates().alias("B"),
        F.col("A.Motivo_de_Rechazo") == F.col("B.MOTIVO"),
        how="left_anti"
    )
)

# Convertir tipo de datos
flujo_one = (
    flujo_one
    .withColumn("Tons_Netas", F.col("Tons_Netas").cast(DoubleType()))
    .withColumn("Tons_Brutas", F.col("Tons_Brutas").cast(DoubleType()))
    .withColumn("Del_Qty", F.col("Del_Qty").cast(DoubleType()))
    .withColumn("Inv_Qty", F.col("Inv_Qty").cast(DoubleType()))
    .withColumn("Ord_Qty", F.col("Ord_Qty").cast(DoubleType()))
    .withColumn("Orig_Qty", F.col("Orig_Qty").cast(DoubleType()))
    .withColumn("CS_GSV", F.col("CS_GSV").cast(DoubleType()))
)

# Calcula cantidad enviada
flujo_one = flujo_one.withColumn(
    "Shp_Qty",
    F.when(
        (F.col("Nro_Transporte").isNull()) |
        (F.trim(F.col("Nro_Transporte").cast("string")) == ""),
        F.lit(0)
    ).otherwise(F.col("Del_Qty"))
)

In [0]:
# Multiplicar toneladas por cantidades
flujo_one = (
    flujo_one
    # ORIGINAL
    .withColumn("Orig_Net_Ton", F.col("Tons_Netas") * F.col("Orig_Qty"))
    .withColumn("Orig_Gross_Ton", F.col("Tons_Brutas") * F.col("Orig_Qty"))
    
    # ORD (Cuanto fue ordenado)
    .withColumn("Ord_Net_Ton", F.col("Tons_Netas") * F.col("Ord_Qty"))
    .withColumn("Ord_Gross_Ton", F.col("Tons_Brutas") * F.col("Ord_Qty"))

    # INV (Facturado)
    .withColumn("Inv_Net_Ton", F.col("Tons_Netas") * F.col("Inv_Qty"))
    .withColumn("Inv_Gross_Ton", F.col("Tons_Brutas") * F.col("Inv_Qty"))

    # SHP (Enviado en camion)
    .withColumn("Shp_Net_Ton", F.col("Tons_Netas") * F.col("Shp_Qty"))
    .withColumn("Shp_Gross_Ton", F.col("Tons_Brutas") * F.col("Shp_Qty"))
)

# Multiplicar GSV por cantidades
flujo_one = (
    flujo_one
    # ORIGINAL
    .withColumn("Orig_GSV", F.col("CS_GSV") * F.col("Orig_Qty")*1000)
    
    # ORD
    .withColumn("Ord_GSV", F.col("CS_GSV") * F.col("Ord_Qty")*1000)
    
    # INV
    .withColumn("Inv_GSV", F.col("CS_GSV") * F.col("Inv_Qty")*1000)

    # SHP
    .withColumn("Shp_GSV", F.col("CS_GSV") * F.col("Shp_Qty")*1000)
)


# Join con la tabla Solicitantes
flujo_one_sol = (
    flujo_one.alias("A")
    .join(
        solicitantes.alias("B"),
        on=["YYYY_MM", "Org_Ventas", "Solicitante", "Sector"],
        how="left_outer"
    )
    .select(
        "A.*",
        "B.Grupo_Cliente",
        "B.Nombre_Grupo_Cliente",
        "B.Pagador",
        F.col("B.Sector").alias("División"),
        #F.col("B.Canal").alias("Canal_Grupo")
    )
    #.filter(
    #    F.col("Grupo_Cliente").isin(
    #    "30094983",
    #    "10901453",
    #    "10901449",
    #    "10901187",
    #    "30278713",
    #    "10901078",
    #    "10901462",
    #    "10901069",
    #    "10901192",
    #    "10901458"
    #)
    #)
)

# Join con la tabla Grupos
flujo_one_group = (
    flujo_one_sol.alias("A")
    .join(
        grupos.alias("B"),
        on=["YYYY_MM", "Org_Ventas", "Grupo_Cliente", "Sector"],
        how="left_outer"
    )
    .select(
        "A.*",
        F.col("B.Canal").alias("Canal_Grupo")
    )
    #.filter(
    #    F.col("Canal_Grupo").isin(
    #    "BPD",
    #    "FARMA",
    #    "FARMA_DIS",
    #    "MAYOREO",
    #    "SSS_REG",
    #    "DEPARTAMENTAL"
    #)
    #)
)


# Join con la tabla MARM
flujo_one_marm = (
    flujo_one_group.alias("A")
    .join(
        marm.alias("B"),
        on="Material",
        how="left_outer"
    )
    .select(
        "A.*",
        F.col("B.CajasxTSP").alias("CS_TSP"),
        F.col("B.CajasxLayer").alias("CS_LAY"),
        F.col("B.CajasxPallet").alias("CS_PAL"),
        F.col("B.VolCS").alias("VOL_CS")
    )
)

# Saca TSP, PAL y LAY por instancia
flujo_one_PAL = flujo_one_marm
for qty_col in [
        #"Del_Qty", 
        #"Inv_Qty", 
        "Ord_Qty", 
        "Orig_Qty", 
        "Shp_Qty"
        ]:
    for sufijo in [
        #("TSP"),
        ("LAY"),
        ("PAL")
    ]:
        flujo_one_PAL = flujo_one_PAL.withColumn(
            f"{qty_col}_{sufijo}",
            F.try_divide(F.col(qty_col), F.col(f"CS_{sufijo}"))
        )

# Saca Volumen por instancia
flujo_one_VOL = flujo_one_PAL
for qty_col in [
        #"Del_Qty", 
        #"Inv_Qty", 
        "Ord_Qty", 
        "Orig_Qty", 
        "Shp_Qty"
        ]:
    
        flujo_one_VOL = flujo_one_VOL.withColumn(
            f"{qty_col}_VOL",
            F.col(qty_col) * F.col("VOL_CS") / 1000
        )

flujo_one_final= flujo_one_VOL

In [0]:
#flujo_one_final.limit(3).display()


Este notebook se debe dividir aqui.Se debe guardar el dataframe "flujo_one_final" como tabla.

Una vez dividido este notebook, se debe crear el dataframe "flujo_one_final" de la tabla "flujo_one_final".

## 1) Procesamiento Eficiente

### Ganado

In [0]:
# Agrupar por día, solicitante y material

agg_exprs = []

for tipo in [
    "Ord",
    "Orig",
    #"Shp"
]:
    agg_exprs.extend([
        F.sum(f"{tipo}_Qty").alias(f"MAT_{tipo}"),
        F.sum(f"{tipo}_GSV").alias(f"MAT_{tipo}_GSV"),
        #F.sum(f"{tipo}_Qty_TSP").alias(f"MAT_{tipo}_Qty_TSP"),
        F.sum(f"{tipo}_Qty_LAY").alias(f"MAT_{tipo}_Qty_LAY"),
        F.sum(f"{tipo}_Qty_PAL").alias(f"MAT_{tipo}_Qty_PAL")])

agg_exprs.extend([
    #F.first("CS_TSP").alias("CS_TSP"),
    F.first("CS_LAY").alias("CS_LAY"),
    F.first("CS_PAL").alias("CS_PAL")
])

PE_Date_Sol_Mat = (
    flujo_one_final
    .groupby(
        "YYYY_MM",
        "Org_Ventas",
        "Fecha_de_Creación_del_Pedido",
        "Grupo_Cliente",
        "Solicitante",
        "Material"
    )
    .agg(*agg_exprs)
)


# Dividir ordenadas / cajas

def calc_pack(df, col_pack, col_cajas):
    return (
        df
        .withColumn(f"{col_pack}_Full", F.floor(F.col(col_pack)))
        .withColumn(f"{col_pack}_CS", F.col(f"{col_pack}_Full") * F.col(col_cajas))
        .withColumn(f"{col_pack}_%", F.try_divide(F.col(f"{col_pack}_CS"), F.col(col_pack) * F.col(col_cajas))
        )
    )

PE_Date_Sol_Mat_med = PE_Date_Sol_Mat

for tipo in [
    #"Shp",
    "Ord", 
    "Orig"
    ]:
    for sufijo in [
        #"TSP",
        "LAY", 
        "PAL"
        ]:

        col_pack = f"MAT_{tipo}_Qty_{sufijo}"

        PE_Date_Sol_Mat_med = calc_pack(
            PE_Date_Sol_Mat_med,
            col_pack,
            f"CS_{sufijo}"
        )

#evaluamos por linea para despues tener su participación

PE_Date_Sol_Mat_eva = PE_Date_Sol_Mat_med

for tipo in [
    "Ord",
    "Orig",
    #"Shp"
]:
    PE_Date_Sol_Mat_eva = (
        PE_Date_Sol_Mat_eva
        .withColumn(f"MAT_{tipo}_PBL_PE_PAL_%", F.when(F.col(f"MAT_{tipo}_Qty_PAL_%") >= 0.8, .004))
        .withColumn(f"MAT_{tipo}_PBL_PE_LAY_%", F.when(F.col(f"MAT_{tipo}_Qty_PAL_%") < 0.8, F.when(F.col(f"MAT_{tipo}_Qty_LAY_%") >= 0.8, .0025)))
        .withColumn(f"MAT_{tipo}_PBL_PE_PAL", F.coalesce(F.try_multiply(F.col(f"MAT_{tipo}_PBL_PE_PAL_%"), F.col(f"MAT_{tipo}_GSV")), F.lit(0)))
        .withColumn(f"MAT_{tipo}_PBL_PE_LAY", F.coalesce(F.try_multiply(F.col(f"MAT_{tipo}_PBL_PE_LAY_%"), F.col(f"MAT_{tipo}_GSV")), F.lit(0)))
        .withColumn(f"MAT_{tipo}_PBL_PE", F.col(f"MAT_{tipo}_PBL_PE_LAY") + F.col(f"MAT_{tipo}_PBL_PE_PAL"))
        .withColumn(f"MAT_{tipo}_PBL_PE_%", F.try_divide(F.col(f"MAT_{tipo}_PBL_PE"), F.col(f"MAT_{tipo}_GSV")))
    )

# Agrupar por día y solicitante
agg_exprs = []

for tipo in [
    "Ord",
    "Orig",
    #"Shp"
]:

    agg_exprs.extend([

        F.sum(f"MAT_{tipo}").alias(f"{tipo}_Total"),
        F.sum(f"MAT_{tipo}_GSV").alias(f"{tipo}_GSV_Total"),

        F.sum(f"MAT_{tipo}_PBL_PE_LAY").alias(f"{tipo}_MAT_PBL_PE_LAY_Total"),
        F.sum(f"MAT_{tipo}_PBL_PE_PAL").alias(f"{tipo}_MAT_PBL_PE_PAL_Total"),
        F.sum(f"MAT_{tipo}_PBL_PE").alias(f"{tipo}_MAT_PBL_PE_Total"),


        #F.sum(f"MAT_{tipo}_Qty_TSP").alias(f"{tipo}_TSP"),
        #F.sum(f"MAT_{tipo}_Qty_TSP_Full").alias(f"{tipo}_TSP_Full"),
        #F.sum(f"MAT_{tipo}_Qty_TSP_CS").alias(f"{tipo}_TSP_CS"),

        F.sum(f"MAT_{tipo}_Qty_PAL").alias(f"{tipo}_PAL"),
        F.sum(f"MAT_{tipo}_Qty_PAL_Full").alias(f"{tipo}_PAL_Full"),
        F.sum(f"MAT_{tipo}_Qty_PAL_CS").alias(f"{tipo}_PAL_CS"),

        F.sum(f"MAT_{tipo}_Qty_LAY").alias(f"{tipo}_LAY"),
        F.sum(f"MAT_{tipo}_Qty_LAY_Full").alias(f"{tipo}_LAY_Full"),
        F.sum(f"MAT_{tipo}_Qty_LAY_CS").alias(f"{tipo}_LAY_CS")

    ])

PE_Date_Sol = (
    PE_Date_Sol_Mat_eva
    .groupBy(
        "YYYY_MM",
        "Org_Ventas",
        "Fecha_de_Creación_del_Pedido",
        "Grupo_Cliente",
        "Solicitante"
    )
    .agg(*agg_exprs)
)


for tipo in [
    "Ord",
    "Orig",
    #"Shp"
]:

    PE_Date_Sol = (
        PE_Date_Sol
        .withColumn(f"{tipo}_PAL_%", F.try_divide(F.col(f"{tipo}_PAL_CS"), F.col(f"{tipo}_Total")))
        .withColumn(f"{tipo}_LAY_%", F.try_divide(F.col(f"{tipo}_LAY_CS"), F.col(f"{tipo}_Total")))
        .withColumn(f"{tipo}_PBL_PE_PAL_%", F.when(F.col(f"{tipo}_PAL_%") >= 0.8, .004))
        .withColumn(f"{tipo}_PBL_PE_LAY_%", F.when(F.col(f"{tipo}_PAL_%") < 0.8, F.when(F.col(f"{tipo}_LAY_%") >= 0.8, .0025)))
        .withColumn(f"{tipo}_PBL_PE_PAL", F.coalesce(F.try_multiply(F.col(f"{tipo}_PBL_PE_PAL_%"), F.col(f"{tipo}_GSV_Total")), F.lit(0)))
        .withColumn(f"{tipo}_PBL_PE_LAY", F.coalesce(F.try_multiply(F.col(f"{tipo}_PBL_PE_LAY_%"), F.col(f"{tipo}_GSV_Total")), F.lit(0)))
        .withColumn(f"{tipo}_PBL_PE", F.col(f"{tipo}_PBL_PE_LAY") + F.col(f"{tipo}_PBL_PE_PAL"))
        .withColumn(f"{tipo}_PBL_PE_%", F.try_divide(F.col(f"{tipo}_PBL_PE"), F.col(f"{tipo}_GSV_Total")))
    )

# Columnas de Date_Sol que SÍ quiero traer desde b
cols_b = [
    c for c in PE_Date_Sol.columns
    if c not in [
        "YYYY_MM",
        "Org_Ventas",
        "Fecha_de_Creación_del_Pedido",
        "Grupo_Cliente",
        "Solicitante",
    ]
]

# Unimos totales con día
PE_Date_Sol_Mat_Total = (
    PE_Date_Sol_Mat_eva.alias("a")
    .join(
        PE_Date_Sol.alias("b"),
        (
            (F.col("a.Fecha_de_Creación_del_Pedido") == F.col("b.Fecha_de_Creación_del_Pedido"))
            & (F.col("a.Org_Ventas") == F.col("b.Org_Ventas"))
            & (F.col("a.Grupo_Cliente") == F.col("b.Grupo_Cliente"))
            & (F.col("a.Solicitante") == F.col("b.Solicitante"))
        ),
        how="left_outer"
    )
    .select(
        "a.*",
        *[F.col(f"b.`{c}`").alias(c) for c in cols_b]
    )
)

#Sacar participación por dia
PE_Date_Sol_Mat_dia = PE_Date_Sol_Mat_Total

for tipo in [
    "Ord",
    "Orig",
    #"Shp"
]:

    PE_Date_Sol_Mat_dia = (
        PE_Date_Sol_Mat_dia
        .withColumn(f"{tipo}_PBL_PE_Part", F.try_divide(F.col(f"MAT_{tipo}_PBL_PE"), F.col(f"{tipo}_MAT_PBL_PE_Total")))
        .withColumn(f"{tipo}_PBL_PE_PAL_Dia", F.coalesce(F.try_multiply(F.col(f"{tipo}_PBL_PE_Part"), F.col(f"{tipo}_PBL_PE_PAL")), F.lit(0)))
        .withColumn(f"{tipo}_PBL_PE_LAY_Dia", F.coalesce(F.try_multiply(F.col(f"{tipo}_PBL_PE_Part"), F.col(f"{tipo}_PBL_PE_LAY")), F.lit(0)))
        .withColumn(f"{tipo}_PBL_PE_Dia", F.col(f"{tipo}_PBL_PE_PAL_Dia") + F.col(f"{tipo}_PBL_PE_LAY_Dia"))
    )


### Error por no consolidar ordenes del día
#### Emula error de la herramienta vieja de PBL para validar impacto // borrar después de tener el impacto


In [0]:
# Agrupar por orden, solicitante y material

agg_exprs = []

for tipo in [
    #"Ord",
    "Orig",
    #"Shp"
]:

    agg_exprs.extend([

        F.sum(f"{tipo}_Qty").alias(f"MAT_{tipo}"),
        F.sum(f"{tipo}_GSV").alias(f"MAT_{tipo}_GSV"),
        #F.sum(f"{tipo}_Qty_TSP").alias(f"MAT_{tipo}_Qty_TSP"),
        F.sum(f"{tipo}_Qty_LAY").alias(f"MAT_{tipo}_Qty_LAY"),
        F.sum(f"{tipo}_Qty_PAL").alias(f"MAT_{tipo}_Qty_PAL")

    ])

agg_exprs.extend([
    #F.first("CS_TSP").alias("CS_TSP"),
    F.first("CS_LAY").alias("CS_LAY"),
    F.first("CS_PAL").alias("CS_PAL")
])

PE_Date_Sol_Ord = (
    flujo_one_final
    .groupby(
        "YYYY_MM",
        "Org_Ventas",
        "Nro_Orden",
        "Grupo_Cliente",
        "Solicitante",
        "Material"
    )
    .agg(*agg_exprs)
)


# Dividir ordenadas / cajas

def calc_pack(df, col_pack, col_cajas):
    return (
        df
        .withColumn(f"{col_pack}_Full", F.floor(F.col(col_pack)))
        .withColumn(f"{col_pack}_CS", F.col(f"{col_pack}_Full") * F.col(col_cajas))
        .withColumn(f"{col_pack}_%", F.try_divide(F.col(f"{col_pack}_CS"), F.col(col_pack) * F.col(col_cajas))
        )
    )

PE_Date_Sol_Ord_med = PE_Date_Sol_Ord 

for tipo in [
    #"Shp",
    #"Ord", 
    "Orig"
    ]:
    for sufijo in [
        #"TSP",
        "LAY", 
        "PAL"
        ]:

        col_pack = f"MAT_{tipo}_Qty_{sufijo}"

        PE_Date_Sol_Ord_med = calc_pack(
            PE_Date_Sol_Ord_med,
            col_pack,
            f"CS_{sufijo}"
        )

#evaluamos por linea para despues tener su participación

PE_Date_Sol_Ord_eva = PE_Date_Sol_Ord_med

for tipo in [
    #"Ord",
    "Orig",
    #"Shp"
]:
    PE_Date_Sol_Ord_eva = (
        PE_Date_Sol_Ord_eva
        .withColumn(f"MAT_{tipo}_PBL_PE_PAL_%", F.when(F.col(f"MAT_{tipo}_Qty_PAL_%") >= 0.8, .004))
        .withColumn(f"MAT_{tipo}_PBL_PE_LAY_%", F.when(F.col(f"MAT_{tipo}_Qty_PAL_%") < 0.8, F.when(F.col(f"MAT_{tipo}_Qty_LAY_%") >= 0.8, .0025)))
        .withColumn(f"MAT_{tipo}_PBL_PE_PAL", F.coalesce(F.try_multiply(F.col(f"MAT_{tipo}_PBL_PE_PAL_%"), F.col(f"MAT_{tipo}_GSV")), F.lit(0)))
        .withColumn(f"MAT_{tipo}_PBL_PE_LAY", F.coalesce(F.try_multiply(F.col(f"MAT_{tipo}_PBL_PE_LAY_%"), F.col(f"MAT_{tipo}_GSV")), F.lit(0)))
        .withColumn(f"MAT_{tipo}_PBL_PE", F.col(f"MAT_{tipo}_PBL_PE_LAY") + F.col(f"MAT_{tipo}_PBL_PE_PAL"))
        .withColumn(f"MAT_{tipo}_PBL_PE_%", F.try_divide(F.col(f"MAT_{tipo}_PBL_PE"), F.col(f"MAT_{tipo}_GSV")))
    )

# Agrupar por día y solicitante
agg_exprs = []

for tipo in [
    #"Ord",
    "Orig",
    #"Shp"
]:

    agg_exprs.extend([

        F.sum(f"MAT_{tipo}").alias(f"{tipo}_Total"),
        F.sum(f"MAT_{tipo}_GSV").alias(f"{tipo}_GSV_Total"),

        F.sum(f"MAT_{tipo}_PBL_PE_LAY").alias(f"{tipo}_MAT_PBL_PE_LAY_Total"),
        F.sum(f"MAT_{tipo}_PBL_PE_PAL").alias(f"{tipo}_MAT_PBL_PE_PAL_Total"),
        F.sum(f"MAT_{tipo}_PBL_PE").alias(f"{tipo}_MAT_PBL_PE_Total"),


        #F.sum(f"MAT_{tipo}_Qty_TSP").alias(f"{tipo}_TSP"),
        #F.sum(f"MAT_{tipo}_Qty_TSP_Full").alias(f"{tipo}_TSP_Full"),
        #F.sum(f"MAT_{tipo}_Qty_TSP_CS").alias(f"{tipo}_TSP_CS"),

        F.sum(f"MAT_{tipo}_Qty_PAL").alias(f"{tipo}_PAL"),
        F.sum(f"MAT_{tipo}_Qty_PAL_Full").alias(f"{tipo}_PAL_Full"),
        F.sum(f"MAT_{tipo}_Qty_PAL_CS").alias(f"{tipo}_PAL_CS"),

        F.sum(f"MAT_{tipo}_Qty_LAY").alias(f"{tipo}_LAY"),
        F.sum(f"MAT_{tipo}_Qty_LAY_Full").alias(f"{tipo}_LAY_Full"),
        F.sum(f"MAT_{tipo}_Qty_LAY_CS").alias(f"{tipo}_LAY_CS")

    ])

PE_Date_Sol_Ord = (
    PE_Date_Sol_Ord_eva
    .groupBy(
        "YYYY_MM",
        "Org_Ventas",
        "Nro_Orden",
        "Grupo_Cliente",
        "Solicitante"
    )
    .agg(*agg_exprs)
)


for tipo in [
    #"Ord",
    "Orig",
    #"Shp"
]:

    PE_Date_Sol_Ord = (
        PE_Date_Sol_Ord
        .withColumn(f"{tipo}_PAL_%", F.try_divide(F.col(f"{tipo}_PAL_CS"), F.col(f"{tipo}_Total")))
        .withColumn(f"{tipo}_LAY_%", F.try_divide(F.col(f"{tipo}_LAY_CS"), F.col(f"{tipo}_Total")))
        .withColumn(f"{tipo}_PBL_PE_PAL_%", F.when(F.col(f"{tipo}_PAL_%") >= 0.8, .004))
        .withColumn(f"{tipo}_PBL_PE_LAY_%", F.when(F.col(f"{tipo}_PAL_%") < 0.8, F.when(F.col(f"{tipo}_LAY_%") >= 0.8, .0025)))
        .withColumn(f"{tipo}_PBL_PE_PAL", F.coalesce(F.try_multiply(F.col(f"{tipo}_PBL_PE_PAL_%"), F.col(f"{tipo}_GSV_Total")), F.lit(0)))
        .withColumn(f"{tipo}_PBL_PE_LAY", F.coalesce(F.try_multiply(F.col(f"{tipo}_PBL_PE_LAY_%"), F.col(f"{tipo}_GSV_Total")), F.lit(0)))
        .withColumn(f"{tipo}_PBL_PE", F.col(f"{tipo}_PBL_PE_LAY") + F.col(f"{tipo}_PBL_PE_PAL"))
        .withColumn(f"{tipo}_PBL_PE_%", F.try_divide(F.col(f"{tipo}_PBL_PE"), F.col(f"{tipo}_GSV_Total")))
    )

# Columnas de Date_Sol que SÍ quiero traer desde b
cols_b = [
    c for c in PE_Date_Sol_Ord.columns
    if c not in [
        "YYYY_MM",
        "Org_Ventas",
        "Nro_Orden",
        "Grupo_Cliente",
        "Solicitante",
    ]
]

# Unimos totales con día
PE_Date_Sol_Ord_Total = (
    PE_Date_Sol_Ord_eva.alias("a")
    .join(
        PE_Date_Sol_Ord.alias("b"),
        (
            (F.col("a.Nro_Orden") == F.col("b.Nro_Orden"))
            & (F.col("a.Org_Ventas") == F.col("b.Org_Ventas"))
            & (F.col("a.Grupo_Cliente") == F.col("b.Grupo_Cliente"))
            & (F.col("a.Solicitante") == F.col("b.Solicitante"))
        ),
        how="left_outer"
    )
    .select(
        "a.*",
        *[F.col(f"b.`{c}`").alias(c) for c in cols_b]
    )
)

#Sacar participación por dia
PE_Date_Sol_Ord_dia = PE_Date_Sol_Ord_Total

for tipo in [
    #"Ord",
    "Orig",
    #"Shp"
]:

    PE_Date_Sol_Ord_dia = (
        PE_Date_Sol_Ord_dia
        .withColumn(f"{tipo}_PBL_PE_Part_Ord", F.try_divide(F.col(f"MAT_{tipo}_PBL_PE"), F.col(f"{tipo}_MAT_PBL_PE_Total")))
        .withColumn(f"{tipo}_PBL_PE_PAL_Dia_Ord", F.coalesce(F.try_multiply(F.col(f"{tipo}_PBL_PE_Part_Ord"), F.col(f"{tipo}_PBL_PE_PAL")), F.lit(0)))
        .withColumn(f"{tipo}_PBL_PE_LAY_Dia_Ord", F.coalesce(F.try_multiply(F.col(f"{tipo}_PBL_PE_Part_Ord"), F.col(f"{tipo}_PBL_PE_LAY")), F.lit(0)))
        .withColumn(f"{tipo}_PBL_PE_Dia_Ord", F.col(f"{tipo}_PBL_PE_PAL_Dia_Ord") + F.col(f"{tipo}_PBL_PE_LAY_Dia_Ord"))
    )


### Enviado

In [0]:
# Agrupar por Transporte, solicitante y material

agg_exprs = []

for tipo in [
    #"Ord",
    #"Orig",
    "Shp"
]:

    agg_exprs.extend([

        F.sum(f"{tipo}_Qty").alias(f"MAT_{tipo}"),
        F.sum(f"{tipo}_GSV").alias(f"MAT_{tipo}_GSV"),
        #F.sum(f"{tipo}_Qty_TSP").alias(f"MAT_{tipo}_Qty_TSP"),
        F.sum(f"{tipo}_Qty_LAY").alias(f"MAT_{tipo}_Qty_LAY"),
        F.sum(f"{tipo}_Qty_PAL").alias(f"MAT_{tipo}_Qty_PAL")

    ])

agg_exprs.extend([
    #F.first("CS_TSP").alias("CS_TSP"),
    F.first("CS_LAY").alias("CS_LAY"),
    F.first("CS_PAL").alias("CS_PAL")
])

PE_DT_Sol_Mat = (
    flujo_one_final
    .groupby(
        #"YYYY_MM",
        "Nro_Transporte",
        "Grupo_Cliente",
        "Solicitante",
        "Material"
    )
    .agg(*agg_exprs)
)


# Dividir ordenadas / cajas

def calc_pack(df, col_pack, col_cajas):
    return (
        df
        .withColumn(f"{col_pack}_Full", F.floor(F.col(col_pack)))
        .withColumn(f"{col_pack}_CS", F.col(f"{col_pack}_Full") * F.col(col_cajas))
        .withColumn(f"{col_pack}_%", F.try_divide(F.col(f"{col_pack}_CS"), F.col(col_pack) * F.col(col_cajas))
        )
    )

PE_DT_Sol_Mat_med = PE_DT_Sol_Mat

for tipo in [
    "Shp"
    #"Ord", 
    #"Orig"
    ]:
    for sufijo in [
        #"TSP",
        "LAY", 
        "PAL"
        ]:

        col_pack = f"MAT_{tipo}_Qty_{sufijo}"

        PE_DT_Sol_Mat_med = calc_pack(
            PE_DT_Sol_Mat_med,
            col_pack,
            f"CS_{sufijo}"
        )

#evaluamos por linea para despues tener su participación

PE_DT_Sol_Mat_eva = PE_DT_Sol_Mat_med

for tipo in [
    #"Ord",
    #"Orig",
    "Shp"
]:
    PE_DT_Sol_Mat_eva = (
        PE_DT_Sol_Mat_eva
        .withColumn(f"MAT_{tipo}_PBL_PE_PAL_%", F.when(F.col(f"MAT_{tipo}_Qty_PAL_%") >= 0.8, .004))
        .withColumn(f"MAT_{tipo}_PBL_PE_LAY_%", F.when(F.col(f"MAT_{tipo}_Qty_PAL_%") < 0.8, F.when(F.col(f"MAT_{tipo}_Qty_LAY_%") >= 0.8, .0025)))
        .withColumn(f"MAT_{tipo}_PBL_PE_PAL", F.coalesce(F.try_multiply(F.col(f"MAT_{tipo}_PBL_PE_PAL_%"), F.col(f"MAT_{tipo}_GSV")), F.lit(0)))
        .withColumn(f"MAT_{tipo}_PBL_PE_LAY", F.coalesce(F.try_multiply(F.col(f"MAT_{tipo}_PBL_PE_LAY_%"), F.col(f"MAT_{tipo}_GSV")), F.lit(0)))
        .withColumn(f"MAT_{tipo}_PBL_PE", F.col(f"MAT_{tipo}_PBL_PE_LAY") + F.col(f"MAT_{tipo}_PBL_PE_PAL"))
        .withColumn(f"MAT_{tipo}_PBL_PE_%", F.try_divide(F.col(f"MAT_{tipo}_PBL_PE"), F.col(f"MAT_{tipo}_GSV")))
    )

# Agrupar por día y solicitante
agg_exprs = []

for tipo in [
    #"Ord",
    #"Orig",
    "Shp"
]:

    agg_exprs.extend([

        F.sum(f"MAT_{tipo}").alias(f"{tipo}_Total"),
        F.sum(f"MAT_{tipo}_GSV").alias(f"{tipo}_GSV_Total"),

        F.sum(f"MAT_{tipo}_PBL_PE_LAY").alias(f"{tipo}_MAT_PBL_PE_LAY_Total"),
        F.sum(f"MAT_{tipo}_PBL_PE_PAL").alias(f"{tipo}_MAT_PBL_PE_PAL_Total"),
        F.sum(f"MAT_{tipo}_PBL_PE").alias(f"{tipo}_MAT_PBL_PE_Total"),

        #F.sum(f"MAT_{tipo}_Qty_TSP").alias(f"{tipo}_TSP"),
        #F.sum(f"MAT_{tipo}_Qty_TSP_Full").alias(f"{tipo}_TSP_Full"),
        #F.sum(f"MAT_{tipo}_Qty_TSP_CS").alias(f"{tipo}_TSP_CS"),

        F.sum(f"MAT_{tipo}_Qty_PAL").alias(f"{tipo}_PAL"),
        F.sum(f"MAT_{tipo}_Qty_PAL_Full").alias(f"{tipo}_PAL_Full"),
        F.sum(f"MAT_{tipo}_Qty_PAL_CS").alias(f"{tipo}_PAL_CS"),

        F.sum(f"MAT_{tipo}_Qty_LAY").alias(f"{tipo}_LAY"),
        F.sum(f"MAT_{tipo}_Qty_LAY_Full").alias(f"{tipo}_LAY_Full"),
        F.sum(f"MAT_{tipo}_Qty_LAY_CS").alias(f"{tipo}_LAY_CS")

    ])

PE_DT_Sol = (
    PE_DT_Sol_Mat_eva
    .groupBy(
        #"YYYY_MM",
        "Nro_Transporte",
        "Grupo_Cliente",
        "Solicitante"
    )
    .agg(*agg_exprs)
)


for tipo in [
    #"Ord",
    #"Orig",
    "Shp"
]:

    PE_DT_Sol = (
        PE_DT_Sol
        .withColumn(f"{tipo}_PAL_%", F.try_divide(F.col(f"{tipo}_PAL_CS"), F.col(f"{tipo}_Total")))
        .withColumn(f"{tipo}_LAY_%", F.try_divide(F.col(f"{tipo}_LAY_CS"), F.col(f"{tipo}_Total")))
        .withColumn(f"{tipo}_PBL_PE_PAL_%", F.when(F.col(f"{tipo}_PAL_%") >= 0.8, .004))
        .withColumn(f"{tipo}_PBL_PE_LAY_%", F.when(F.col(f"{tipo}_PAL_%") < 0.8, F.when(F.col(f"{tipo}_LAY_%") >= 0.8, .0025)))
        .withColumn(f"{tipo}_PBL_PE_PAL", F.coalesce(F.try_multiply(F.col(f"{tipo}_PBL_PE_PAL_%"), F.col(f"{tipo}_GSV_Total")), F.lit(0)))
        .withColumn(f"{tipo}_PBL_PE_LAY", F.coalesce(F.try_multiply(F.col(f"{tipo}_PBL_PE_LAY_%"), F.col(f"{tipo}_GSV_Total")), F.lit(0)))
        .withColumn(f"{tipo}_PBL_PE", F.col(f"{tipo}_PBL_PE_LAY") + F.col(f"{tipo}_PBL_PE_PAL"))
        .withColumn(f"{tipo}_PBL_PE_%", F.try_divide(F.col(f"{tipo}_PBL_PE"), F.col(f"{tipo}_GSV_Total")))
    )

# Columnas de Date_Sol que SÍ quiero traer desde b
cols_b = [
    c for c in PE_DT_Sol.columns
    if c not in [
        #"YYYY_MM",
        "Nro_Transporte",
        "Grupo_Cliente",
        "Solicitante"
    ]
]

# Unimos totales con día
PE_DT_Sol_Mat_Total = (
    PE_DT_Sol_Mat_eva.alias("a")
    .join(
        PE_DT_Sol.alias("b"),
        (
            (F.col("a.Nro_Transporte") == F.col("b.Nro_Transporte"))
            & (F.col("a.Grupo_Cliente") == F.col("b.Grupo_Cliente"))
            & (F.col("a.Solicitante") == F.col("b.Solicitante"))
        ),
        how="left_outer"
    )
    .select(
        "a.*",
        *[F.col(f"b.`{c}`").alias(c) for c in cols_b]
    )
)

#Sacar participación por dia
PE_DT_Sol_Mat_dia = PE_DT_Sol_Mat_Total

for tipo in [
    #"Ord",
    #"Orig",
    "Shp"
]:

    PE_DT_Sol_Mat_dia = (
       PE_DT_Sol_Mat_dia
        .withColumn(f"{tipo}_PBL_PE_Part", F.try_divide(F.col(f"MAT_{tipo}_PBL_PE"), F.col(f"{tipo}_MAT_PBL_PE_Total")))
        .withColumn(f"{tipo}_PBL_PE_PAL_Dia", F.coalesce(F.try_multiply(F.col(f"{tipo}_PBL_PE_Part"), F.col(f"{tipo}_PBL_PE_PAL")), F.lit(0)))
        .withColumn(f"{tipo}_PBL_PE_LAY_Dia", F.coalesce(F.try_multiply(F.col(f"{tipo}_PBL_PE_Part"), F.col(f"{tipo}_PBL_PE_LAY")), F.lit(0)))
        .withColumn(f"{tipo}_PBL_PE_Dia", F.col(f"{tipo}_PBL_PE_PAL_Dia") + F.col(f"{tipo}_PBL_PE_LAY_Dia"))
    )


## 2) Envío Eficiente

### Ganado

In [0]:
# Agrupar por día y solicitante

agg_exprs = []

for tipo in [
    "Ord",
    "Orig",
    #"Shp"
]:

    agg_exprs.extend([

        F.sum(f"{tipo}_Qty").alias(f"Day_{tipo}"),
        F.sum(f"{tipo}_GSV").alias(f"Day_{tipo}_GSV"),
        #F.sum(f"{tipo}_Qty_TSP").alias(f"Day_{tipo}_Qty_TSP"),
        F.sum(f"{tipo}_Qty_PAL").alias(f"Day_{tipo}_Qty_PAL"),
        F.sum(f"{tipo}_Gross_Ton").alias(f"Day_{tipo}_Gross_Ton"),
        F.sum(f"{tipo}_Qty_VOL").alias(f"Day_{tipo}_Qty_VOL")

    ])


EE_Date_Sol = (
    flujo_one_final
    .groupby(
        "YYYY_MM",
        "Org_Ventas",
        "Fecha_de_Creación_del_Pedido",
        "Grupo_Cliente",
        "Solicitante",
    )
    .agg(*agg_exprs)
)


# Calcula TF y evalua

EE_Date_Sol_med = EE_Date_Sol


def calc_pack(df, prefijo):
    return (
        df
        #Trailer
        .withColumn(f"{prefijo}_TFTR_TON", F.try_divide(F.col(f"{prefijo}_Gross_Ton"), F.lit(25.5)))
        .withColumn(f"{prefijo}_TFTR_PAL", F.try_divide(F.col(f"{prefijo}_Qty_PAL"), F.lit(30)))
        .withColumn(f"{prefijo}_TFTR_VOL", F.try_divide(F.col(f"{prefijo}_Qty_VOL"), F.lit(70)))
        .withColumn(f"{prefijo}_TFTR_Max", F.greatest(F.col(f"{prefijo}_TFTR_TON"), F.col(f"{prefijo}_TFTR_PAL"), F.col(f"{prefijo}_TFTR_VOL")))
        .withColumn(f"{prefijo}_Trailers", F.ceil(F.col(f"{prefijo}_TFTR_MAX")))
        .withColumn(f"{prefijo}_TFTR_over80", F.ceil(F.col(f"{prefijo}_TFTR_MAX"))*F.lit(0.8))
        .withColumn(f"{prefijo}_TFTR_over75", F.ceil(F.col(f"{prefijo}_TFTR_MAX"))*F.lit(0.75))
        .withColumn(f"{prefijo}_TFTR_FULL", F.floor(F.col(f"{prefijo}_TFTR_MAX")))
        .withColumn(f"{prefijo}_TFTR_REST", F.col(f"{prefijo}_TFTR_MAX")-F.col(f"{prefijo}_TFTR_FULL"))
        .withColumn(f"{prefijo}_TFTR_PBL_Cap", F.when(F.col(f"{prefijo}_TFTR_Max") >= F.col(f"{prefijo}_TFTR_over80"), F.lit(0.008)).when(F.col(f"{prefijo}_TFTR_Max") >= F.col(f"{prefijo}_TFTR_over75"), F.lit(0.0075)).otherwise(F.lit(0)))
        .withColumn(f"{prefijo}_TFTR_PBL_Full", F.try_divide(F.col(f"{prefijo}_TFTR_FULL"), F.col(f"{prefijo}_TFTR_Max"))*F.lit(0.008))
        .withColumn(f"{prefijo}_TFTR_PBL_Rest", F.try_divide(F.col(f"{prefijo}_TFTR_REST"), F.col(f"{prefijo}_TFTR_Max"))* F.when(F.col(f"{prefijo}_TFTR_REST") >= F.lit(0.8), F.lit(0.008)).when(F.col(f"{prefijo}_TFTR_REST") >= F.lit(0.75), F.lit(0.0075)).otherwise(F.lit(0)))
        .withColumn(f"{prefijo}_TFTR_PBL_FullRest", F.col(f"{prefijo}_TFTR_PBL_Full") + F.col(f"{prefijo}_TFTR_PBL_Rest"))
        .withColumn(f"{prefijo}_TFTR_PBL_Old", F.when(F.col(f"{prefijo}_TFTR_Max") >= F.lit(0.8), F.lit(0.008)).when(F.col(f"{prefijo}_TFTR_Max") >=  F.lit(0.75), F.lit(0.0075)).otherwise(F.lit(0)))

        #Thorton
        .withColumn(f"{prefijo}_TFTH_TON", F.try_divide(F.col(f"{prefijo}_Gross_Ton"), F.lit(13.5)))
        .withColumn(f"{prefijo}_TFTH_PAL", F.try_divide(F.col(f"{prefijo}_Qty_PAL"), F.lit(12)))
        .withColumn(f"{prefijo}_TFTH_VOL", F.try_divide(F.col(f"{prefijo}_Qty_VOL"), F.lit(20)))
        .withColumn(f"{prefijo}_TFTH_Max", F.greatest(F.col(f"{prefijo}_TFTH_TON"), F.col(f"{prefijo}_TFTH_PAL"), F.col(f"{prefijo}_TFTH_VOL")))
        .withColumn(f"{prefijo}_Thortons", F.ceil(F.col(f"{prefijo}_TFTH_MAX")))
        .withColumn(f"{prefijo}_TFTH_over80", F.ceil(F.col(f"{prefijo}_TFTH_MAX"))*F.lit(0.8))
        .withColumn(f"{prefijo}_TFTH_over75", F.ceil(F.col(f"{prefijo}_TFTH_MAX"))*F.lit(0.75))
        .withColumn(f"{prefijo}_TFTH_FULL", F.floor(F.col(f"{prefijo}_TFTH_MAX")))
        .withColumn(f"{prefijo}_TFTH_REST", F.col(f"{prefijo}_TFTH_MAX")-F.col(f"{prefijo}_TFTH_FULL"))
        .withColumn(f"{prefijo}_TFTH_PBL_Cap", F.when(F.col(f"{prefijo}_TFTH_Max") >= F.col(f"{prefijo}_TFTH_over80"), F.lit(0.008)).when(F.col(f"{prefijo}_TFTH_Max") >= F.col(f"{prefijo}_TFTH_over75"), F.lit(0.0075)).otherwise(F.lit(0)))
        .withColumn(f"{prefijo}_TFTH_PBL_Full", F.try_divide(F.col(f"{prefijo}_TFTH_FULL"), F.col(f"{prefijo}_TFTH_Max"))*F.lit(0.008))
        .withColumn(f"{prefijo}_TFTH_PBL_Rest", F.try_divide(F.col(f"{prefijo}_TFTH_REST"), F.col(f"{prefijo}_TFTH_Max"))* F.when(F.col(f"{prefijo}_TFTH_REST") >= F.lit(0.8), F.lit(0.008)).when(F.col(f"{prefijo}_TFTH_REST") >= F.lit(0.75), F.lit(0.0075)).otherwise(F.lit(0)))
        .withColumn(f"{prefijo}_TFTH_PBL_FullRest", F.col(f"{prefijo}_TFTH_PBL_Full") + F.col(f"{prefijo}_TFTH_PBL_Rest"))
        .withColumn(f"{prefijo}_TFTH_PBL_Old", F.when(F.col(f"{prefijo}_TFTH_Max") >= F.lit(0.8), F.lit(0.008)).when(F.col(f"{prefijo}_TFTH_Max") >=  F.lit(0.75), F.lit(0.0075)).otherwise(F.lit(0)))
        
        
    )

for tipo in [
    #"Shp",
    "Ord",
    "Orig"
    ]:
    EE_Date_Sol_med = calc_pack(
        EE_Date_Sol_med,
        f"Day_{tipo}"
    )




In [0]:

#EE_Date_Sol_med.display()  

### Enviado

In [0]:
# Agrupar por transporte y solicitante

agg_exprs = []

for tipo in [
    #"Ord",
    #"Orig",
    "Shp"
]:

    agg_exprs.extend([

        F.sum(f"{tipo}_Qty").alias(f"Day_{tipo}"),
        F.sum(f"{tipo}_GSV").alias(f"Day_{tipo}_GSV"),
        #F.sum(f"{tipo}_Qty_TSP").alias(f"Day_{tipo}_Qty_TSP"),
        F.sum(f"{tipo}_Qty_PAL").alias(f"Day_{tipo}_Qty_PAL"),
        F.sum(f"{tipo}_Gross_Ton").alias(f"Day_{tipo}_Gross_Ton"),
        F.sum(f"{tipo}_Qty_VOL").alias(f"Day_{tipo}_Qty_VOL")

    ])


EE_Date_Sol_DT = (
    flujo_one_final
    .groupby(
        "Nro_Transporte",
        "Grupo_Cliente",
        "Solicitante",
    )
    .agg(*agg_exprs)
)


# Calcula TF y evalua

EE_Date_Sol_med_DT = EE_Date_Sol_DT


def calc_pack(df, prefijo):
    return (
        df
        #Trailer
        .withColumn(f"{prefijo}_TFTR_TON", F.try_divide(F.col(f"{prefijo}_Gross_Ton"), F.lit(25.5)))
        .withColumn(f"{prefijo}_TFTR_PAL", F.try_divide(F.col(f"{prefijo}_Qty_PAL"), F.lit(30)))
        .withColumn(f"{prefijo}_TFTR_VOL", F.try_divide(F.col(f"{prefijo}_Qty_VOL"), F.lit(70)))
        .withColumn(f"{prefijo}_TFTR_Max", F.greatest(F.col(f"{prefijo}_TFTR_TON"), F.col(f"{prefijo}_TFTR_PAL"), F.col(f"{prefijo}_TFTR_VOL")))
        .withColumn(f"{prefijo}_TFTR_over80", F.ceil(F.col(f"{prefijo}_TFTR_MAX"))*F.lit(0.8))
        .withColumn(f"{prefijo}_TFTR_over75", F.ceil(F.col(f"{prefijo}_TFTR_MAX"))*F.lit(0.75))
        .withColumn(f"{prefijo}_TFTR_FULL", F.floor(F.col(f"{prefijo}_TFTR_MAX")))
        .withColumn(f"{prefijo}_TFTR_REST", F.col(f"{prefijo}_TFTR_MAX")-F.col(f"{prefijo}_TFTR_FULL"))
        .withColumn(f"{prefijo}_TFTR_PBL_Cap", F.when(F.col(f"{prefijo}_TFTR_Max") >= F.col(f"{prefijo}_TFTR_over80"), F.lit(0.008)).when(F.col(f"{prefijo}_TFTR_Max") >= F.col(f"{prefijo}_TFTR_over75"), F.lit(0.0075)).otherwise(F.lit(0)))
        .withColumn(f"{prefijo}_TFTR_PBL_Full", F.try_divide(F.col(f"{prefijo}_TFTR_FULL"), F.col(f"{prefijo}_TFTR_Max"))*F.lit(0.008))
        .withColumn(f"{prefijo}_TFTR_PBL_Rest", F.try_divide(F.col(f"{prefijo}_TFTR_REST"), F.col(f"{prefijo}_TFTR_Max"))* F.when(F.col(f"{prefijo}_TFTR_REST") >= F.lit(0.8), F.lit(0.008)).when(F.col(f"{prefijo}_TFTR_REST") >= F.lit(0.75), F.lit(0.0075)).otherwise(F.lit(0)))
        .withColumn(f"{prefijo}_TFTR_PBL_FullRest", F.col(f"{prefijo}_TFTR_PBL_Full") + F.col(f"{prefijo}_TFTR_PBL_Rest"))
        .withColumn(f"{prefijo}_TFTR_PBL_Old", F.when(F.col(f"{prefijo}_TFTR_Max") >= F.lit(0.8), F.lit(0.008)).when(F.col(f"{prefijo}_TFTR_Max") >=  F.lit(0.75), F.lit(0.0075)).otherwise(F.lit(0)))

        #Thorton
        .withColumn(f"{prefijo}_TFTH_TON", F.try_divide(F.col(f"{prefijo}_Gross_Ton"), F.lit(13.5)))
        .withColumn(f"{prefijo}_TFTH_PAL", F.try_divide(F.col(f"{prefijo}_Qty_PAL"), F.lit(12)))
        .withColumn(f"{prefijo}_TFTH_VOL", F.try_divide(F.col(f"{prefijo}_Qty_VOL"), F.lit(20)))
        .withColumn(f"{prefijo}_TFTH_Max", F.greatest(F.col(f"{prefijo}_TFTH_TON"), F.col(f"{prefijo}_TFTH_PAL"), F.col(f"{prefijo}_TFTH_VOL")))
        .withColumn(f"{prefijo}_TFTH_over80", F.ceil(F.col(f"{prefijo}_TFTH_MAX"))*F.lit(0.8))
        .withColumn(f"{prefijo}_TFTH_over75", F.ceil(F.col(f"{prefijo}_TFTH_MAX"))*F.lit(0.75))
        .withColumn(f"{prefijo}_TFTH_FULL", F.floor(F.col(f"{prefijo}_TFTH_MAX")))
        .withColumn(f"{prefijo}_TFTH_REST", F.col(f"{prefijo}_TFTH_MAX")-F.col(f"{prefijo}_TFTH_FULL"))
        .withColumn(f"{prefijo}_TFTH_PBL_Cap", F.when(F.col(f"{prefijo}_TFTH_Max") >= F.col(f"{prefijo}_TFTH_over80"), F.lit(0.008)).when(F.col(f"{prefijo}_TFTH_Max") >= F.col(f"{prefijo}_TFTH_over75"), F.lit(0.0075)).otherwise(F.lit(0)))
        .withColumn(f"{prefijo}_TFTH_PBL_Full", F.try_divide(F.col(f"{prefijo}_TFTH_FULL"), F.col(f"{prefijo}_TFTH_Max"))*F.lit(0.008))
        .withColumn(f"{prefijo}_TFTH_PBL_Rest", F.try_divide(F.col(f"{prefijo}_TFTH_REST"), F.col(f"{prefijo}_TFTH_Max"))* F.when(F.col(f"{prefijo}_TFTH_REST") >= F.lit(0.8), F.lit(0.008)).when(F.col(f"{prefijo}_TFTH_REST") >= F.lit(0.75), F.lit(0.0075)).otherwise(F.lit(0)))
        .withColumn(f"{prefijo}_TFTH_PBL_FullRest", F.col(f"{prefijo}_TFTH_PBL_Full") + F.col(f"{prefijo}_TFTH_PBL_Rest"))
        .withColumn(f"{prefijo}_TFTH_PBL_Old", F.when(F.col(f"{prefijo}_TFTH_Max") >= F.lit(0.8), F.lit(0.008)).when(F.col(f"{prefijo}_TFTH_Max") >=  F.lit(0.75), F.lit(0.0075)).otherwise(F.lit(0)))
        
        
    )

for tipo in [
    "Shp",
    #"Ord",
    #"Orig"
    ]:
    EE_Date_Sol_med_DT = calc_pack(
        EE_Date_Sol_med_DT,
        f"Day_{tipo}"
    )




##F) Unir Al flujo

### Procesamiento eficiente

In [0]:
flujo_one_PE = (
    flujo_one_final.alias("a")
    .join(
        PE_Date_Sol_Mat_dia.alias("b"),
        (
            (F.col("a.Fecha_de_Creación_del_Pedido") == F.col("b.Fecha_de_Creación_del_Pedido"))
            & (F.col("a.Org_Ventas") == F.col("b.Org_Ventas"))
            & (F.col("a.Grupo_Cliente") == F.col("b.Grupo_Cliente"))
            & (F.col("a.Solicitante") == F.col("b.Solicitante"))
            & (F.col("a.Material") == F.col("b.Material"))
        ),
        how="left_outer"
    )
    .select(
        "a.*",

        # ORD
        F.coalesce(F.try_multiply(F.col("b.MAT_Ord_Qty_PAL_%"), F.col("a.Ord_Qty")), F.lit(0)).alias("ORD_CS_FULL_PAL"),
        F.coalesce(F.try_multiply(F.col("b.MAT_Ord_Qty_LAY_%"), F.col("a.Ord_Qty")), F.lit(0)).alias("ORD_CS_FULL_LAY"),
        F.coalesce(F.try_multiply(F.try_divide(F.col("a.Ord_Qty"), F.col("b.MAT_Ord")), F.col("b.Ord_PBL_PE_PAL_Dia")), F.lit(0)).alias("ORD_PBL_PE_PAL_Line"),
        F.coalesce(F.try_multiply(F.try_divide(F.col("a.Ord_Qty"), F.col("b.MAT_Ord")), F.col("b.Ord_PBL_PE_LAY_Dia")), F.lit(0)).alias("ORD_PBL_PE_LAY_Line"),

        # ORIG
        F.coalesce(F.try_multiply(F.col("b.MAT_Orig_Qty_PAL_%"), F.col("a.Orig_Qty")), F.lit(0)).alias("ORIG_CS_FULL_PAL"),
        F.coalesce(F.try_multiply(F.col("b.MAT_Orig_Qty_LAY_%"), F.col("a.Orig_Qty")), F.lit(0)).alias("ORIG_CS_FULL_LAY"),
        F.coalesce(F.try_multiply(F.try_divide(F.col("a.Orig_Qty"), F.col("b.MAT_Orig")), F.col("b.Orig_PBL_PE_PAL_Dia")), F.lit(0)).alias("ORIG_PBL_PE_PAL_Line"),
        F.coalesce(F.try_multiply(F.try_divide(F.col("a.Orig_Qty"), F.col("b.MAT_Orig")), F.col("b.Orig_PBL_PE_LAY_Dia")), F.lit(0)).alias("ORIG_PBL_PE_LAY_Line")

        # SHP
        #,F.coalesce(F.try_multiply(F.col("b.MAT_Shp_Qty_PAL_%"), F.col("a.Shp_Qty")), F.lit(0)).alias("SHP_CS_FULL_PAL")
        #,F.coalesce(F.try_multiply(F.col("b.MAT_Shp_Qty_LAY_%"), F.col("a.Shp_Qty")), F.lit(0)).alias("SHP_CS_FULL_LAY")
        #,F.coalesce(F.try_multiply(F.try_divide(F.col("a.Shp_Qty"), F.col("b.MAT_Shp")), F.col("b.Shp_PBL_PE_PAL_Dia")), F.lit(0)).alias("SHP_PBL_PE_PAL_Line")
        #,F.coalesce(F.try_multiply(F.try_divide(F.col("a.Shp_Qty"), F.col("b.MAT_Shp")), F.col("b.Shp_PBL_PE_LAY_Dia")), F.lit(0)).alias("SHP_PBL_PE_LAY_Line")
    )
)

In [0]:
flujo_one_PE_Ord = (
    flujo_one_PE.alias("a")
    .join(
        PE_Date_Sol_Ord_dia.alias("b"),
        (
            (F.col("a.Nro_Orden") == F.col("b.Nro_Orden"))
            & (F.col("a.Org_Ventas") == F.col("b.Org_Ventas"))
            & (F.col("a.Grupo_Cliente") == F.col("b.Grupo_Cliente"))
            & (F.col("a.Solicitante") == F.col("b.Solicitante"))
            & (F.col("a.Material") == F.col("b.Material"))
        ),
        how="left_outer"
    )
    .select(
        "a.*",
        # ORIG
        F.coalesce(F.try_multiply(F.col("b.MAT_Orig_Qty_PAL_%"), F.col("a.Orig_Qty")), F.lit(0)).alias("ORIG_CS_FULL_PAL_Ord"),
        F.coalesce(F.try_multiply(F.col("b.MAT_Orig_Qty_LAY_%"), F.col("a.Orig_Qty")), F.lit(0)).alias("ORIG_CS_FULL_LAY_Ord"),
        F.coalesce(F.try_multiply(F.try_divide(F.col("a.Orig_Qty"), F.col("b.MAT_Orig")), F.col("b.Orig_PBL_PE_PAL_Dia_Ord")), F.lit(0)).alias("ORIG_PBL_PE_PAL_Line_Ord"),
        F.coalesce(F.try_multiply(F.try_divide(F.col("a.Orig_Qty"), F.col("b.MAT_Orig")), F.col("b.Orig_PBL_PE_LAY_Dia_Ord")), F.lit(0)).alias("ORIG_PBL_PE_LAY_Line_Ord")

        
    )
)

In [0]:
flujo_one_PE_DT = (
    flujo_one_PE_Ord.alias("a")
    .join(
        PE_DT_Sol_Mat_dia.alias("b"),
        (
            (F.col("a.Nro_Transporte") == F.col("b.Nro_Transporte"))
            & (F.col("a.Grupo_Cliente") == F.col("b.Grupo_Cliente"))
            & (F.col("a.Solicitante") == F.col("b.Solicitante"))
            & (F.col("a.Material") == F.col("b.Material"))
        ),
        how="left_outer"
    )
    .select(
        "a.*",

        # SHP
        F.coalesce(F.try_multiply(F.col("b.MAT_Shp_Qty_PAL_%"), F.col("a.Shp_Qty")), F.lit(0)).alias("SHP_CS_FULL_PAL")
        ,F.coalesce(F.try_multiply(F.col("b.MAT_Shp_Qty_LAY_%"), F.col("a.Shp_Qty")), F.lit(0)).alias("SHP_CS_FULL_LAY")
        ,F.coalesce(F.try_multiply(F.try_divide(F.col("a.Shp_Qty"), F.col("b.MAT_Shp")), F.col("b.Shp_PBL_PE_PAL_Dia")), F.lit(0)).alias("SHP_PBL_PE_PAL_Line")
        ,F.coalesce(F.try_multiply(F.try_divide(F.col("a.Shp_Qty"), F.col("b.MAT_Shp")), F.col("b.Shp_PBL_PE_LAY_Dia")), F.lit(0)).alias("SHP_PBL_PE_LAY_Line")
    )
)

In [0]:
#flujo_one_PE_DT.limit(10).display() 

### Envío Eficiente

In [0]:
cols_EE = []

for tipo_gsv in ["Ord", "Orig"]:
    for tipo_pbl in ["TFTR", "TFTH"]:

        cols_EE.extend([
            F.coalesce(F.try_multiply(F.col(f"b.Day_{tipo_gsv}_{tipo_pbl}_PBL_Cap"), F.col(f"a.{tipo_gsv}_GSV")), F.lit(0)).alias(f"{tipo_gsv}_{tipo_pbl}_PBL_Cap"),
            F.coalesce(F.try_multiply(F.col(f"b.Day_{tipo_gsv}_{tipo_pbl}_PBL_FullRest"), F.col(f"a.{tipo_gsv}_GSV")), F.lit(0)).alias(f"{tipo_gsv}_{tipo_pbl}_PBL_FullRest"),
            F.greatest(F.coalesce(F.try_multiply(F.col(f"b.Day_{tipo_gsv}_{tipo_pbl}_PBL_Cap"), F.col(f"a.{tipo_gsv}_GSV")), F.lit(0)), F.coalesce(F.try_multiply(F.col(f"b.Day_{tipo_gsv}_{tipo_pbl}_PBL_FullRest"), F.col(f"a.{tipo_gsv}_GSV")), F.lit(0))).alias(f"{tipo_gsv}_{tipo_pbl}_PBL_CAPFULL"),
            F.coalesce(F.try_multiply(F.col(f"b.Day_{tipo_gsv}_{tipo_pbl}_PBL_Old"), F.col(f"a.{tipo_gsv}_GSV")), F.lit(0)).alias(f"{tipo_gsv}_{tipo_pbl}_PBL_Old"),
            F.coalesce(F.try_multiply(F.col(f"b.Day_{tipo_gsv}_{tipo_pbl}_MAX"), F.col(f"a.{tipo_gsv}_QTY")), F.lit(0)).alias(f"{tipo_gsv}_{tipo_pbl}_MAX_CAP")
        ])

flujo_one_EE = (
    flujo_one_PE_DT.alias("a")
    .join(
        EE_Date_Sol_med.alias("b"),
        (
            (F.col("a.Fecha_de_Creación_del_Pedido") == F.col("b.Fecha_de_Creación_del_Pedido"))
            & (F.col("a.Org_Ventas") == F.col("b.Org_Ventas"))
            & (F.col("a.Grupo_Cliente") == F.col("b.Grupo_Cliente"))
            & (F.col("a.Solicitante") == F.col("b.Solicitante"))
        ),
        how="left_outer"
    )
    .select(
        "a.*",
        *cols_EE
    )
)


#flujo_one_EE.limit(10).display()     

In [0]:
cols_EE = []

for tipo_gsv in ["Shp"]:
    for tipo_pbl in ["TFTR", "TFTH"]:

        cols_EE.extend([
            F.coalesce(F.try_multiply(F.col(f"b.Day_{tipo_gsv}_{tipo_pbl}_PBL_Cap"), F.col(f"a.{tipo_gsv}_GSV")), F.lit(0)).alias(f"{tipo_gsv}_{tipo_pbl}_PBL_Cap"),
            F.coalesce(F.try_multiply(F.col(f"b.Day_{tipo_gsv}_{tipo_pbl}_PBL_FullRest"), F.col(f"a.{tipo_gsv}_GSV")), F.lit(0)).alias(f"{tipo_gsv}_{tipo_pbl}_PBL_FullRest"),
            F.greatest(F.coalesce(F.try_multiply(F.col(f"b.Day_{tipo_gsv}_{tipo_pbl}_PBL_Cap"), F.col(f"a.{tipo_gsv}_GSV")), F.lit(0)), F.coalesce(F.try_multiply(F.col(f"b.Day_{tipo_gsv}_{tipo_pbl}_PBL_FullRest"), F.col(f"a.{tipo_gsv}_GSV")), F.lit(0))).alias(f"{tipo_gsv}_{tipo_pbl}_PBL_CAPFULL"),
            F.coalesce(F.try_multiply(F.col(f"b.Day_{tipo_gsv}_{tipo_pbl}_PBL_Old"), F.col(f"a.{tipo_gsv}_GSV")), F.lit(0)).alias(f"{tipo_gsv}_{tipo_pbl}_PBL_Old"),
            F.coalesce(F.try_multiply(F.col(f"b.Day_{tipo_gsv}_{tipo_pbl}_MAX"), F.col(f"a.{tipo_gsv}_QTY")), F.lit(0)).alias(f"{tipo_gsv}_{tipo_pbl}_MAX_CAP")

        ])


flujo_one_EE_DT = (
    flujo_one_EE.alias("a")
    .join(
        EE_Date_Sol_med_DT.alias("b"),
        (
            (F.col("a.Nro_Transporte") == F.col("b.Nro_Transporte"))
            & (F.col("a.Grupo_Cliente") == F.col("b.Grupo_Cliente"))
            & (F.col("a.Solicitante") == F.col("b.Solicitante"))
        ),
        how="left_outer"
    )
    .select(
        "a.*",
        *cols_EE
    )
)

# Tabla final


In [0]:
Tabla_Final = flujo_one_EE_DT

# Persisting Table

In [0]:
%sql
use schema mx_unilever_pbl

In [0]:
(Tabla_Final.write.mode("overwrite").format("delta").option("header", "true")
 .option("overwriteSchema", "true").option("encoding", "UTF-8")
 .save("/mnt/adls/SupplyChain/PBL/PBL_Efficiency_dev"))


spark.sql("""CREATE TABLE IF NOT EXISTS PBL_Efficiency
USING DELTA
OPTIONS
(
PATH 'dbfs:/mnt/adls/SupplyChain/PBL/PBL_Efficiency_dev'
)""")

DataFrame[]

In [0]:
%sql
--SELECT * FROM mx_unilever_pbl.PBL_Efficiency LIMIT 10;